# 01 - Download Dataset

In this notebook I download the two datasets, check what is inside them and save the three columns this project uses.

Both datasets are from Open Power System Data (OPSD):
- **Time series** (version 2020-10-06): hourly electricity load for European countries
- **Weather data** (version 2020-09-16): hourly temperature and radiation for European countries

The files are saved in `datasets/raw`.

### 1. Import libraries

In [1]:
import os
import urllib.request

import pandas as pd

### 2. Download the files
If a file is already downloaded, it is not downloaded again.

In [2]:
raw_folder = "../datasets/raw"
os.makedirs(raw_folder, exist_ok=True)

load_url = "https://data.open-power-system-data.org/time_series/2020-10-06/time_series_60min_singleindex.csv"
weather_url = "https://data.open-power-system-data.org/weather_data/2020-09-16/weather_data.csv"

load_file = raw_folder + "/time_series_60min_singleindex.csv"
weather_file = raw_folder + "/weather_data.csv"

In [3]:
if not os.path.exists(load_file):
    print("Downloading load data (about 130 MB)...")
    urllib.request.urlretrieve(load_url, load_file)

if not os.path.exists(weather_file):
    print("Downloading weather data (about 230 MB)...")
    urllib.request.urlretrieve(weather_url, weather_file)

print("Load file size   :", round(os.path.getsize(load_file) / 1000000), "MB")
print("Weather file size:", round(os.path.getsize(weather_file) / 1000000), "MB")

Load file size   : 130 MB
Weather file size: 234 MB


### 3. What is inside the load file

In [4]:
load_data = pd.read_csv(load_file)

print("Rows:", load_data.shape[0])
print("Columns:", load_data.shape[1])
print("From", load_data["utc_timestamp"].iloc[0], "to", load_data["utc_timestamp"].iloc[-1])

Rows: 50401
Columns: 300
From 2014-12-31T23:00:00Z to 2020-09-30T23:00:00Z


In [5]:
# column names look like: country_area_variable_source
# GB_GBN = Great Britain, GB_NIR = Northern Ireland, GB_UKM = whole United Kingdom
for col in load_data.columns:
    if col.startswith("GB_"):
        print(col)

GB_GBN_load_actual_entsoe_transparency
GB_GBN_load_forecast_entsoe_transparency
GB_GBN_price_day_ahead
GB_GBN_solar_capacity
GB_GBN_solar_generation_actual
GB_GBN_solar_profile
GB_GBN_wind_capacity
GB_GBN_wind_generation_actual
GB_GBN_wind_profile
GB_GBN_wind_offshore_capacity
GB_GBN_wind_offshore_generation_actual
GB_GBN_wind_offshore_profile
GB_GBN_wind_onshore_capacity
GB_GBN_wind_onshore_generation_actual
GB_GBN_wind_onshore_profile
GB_NIR_load_actual_entsoe_transparency
GB_NIR_load_forecast_entsoe_transparency
GB_NIR_solar_capacity
GB_NIR_wind_onshore_capacity
GB_NIR_wind_onshore_generation_actual
GB_UKM_load_actual_entsoe_transparency
GB_UKM_load_forecast_entsoe_transparency
GB_UKM_solar_capacity
GB_UKM_solar_generation_actual
GB_UKM_wind_capacity
GB_UKM_wind_generation_actual
GB_UKM_wind_offshore_capacity
GB_UKM_wind_offshore_generation_actual
GB_UKM_wind_onshore_capacity
GB_UKM_wind_onshore_generation_actual


### 4. What is inside the weather file

In [6]:
weather_data = pd.read_csv(weather_file)

print("Rows:", weather_data.shape[0])
print("Columns:", weather_data.shape[1])
print("From", weather_data["utc_timestamp"].iloc[0], "to", weather_data["utc_timestamp"].iloc[-1])

Rows: 350640
Columns: 85
From 1980-01-01T00:00:00Z to 2019-12-31T23:00:00Z


In [7]:
for col in weather_data.columns:
    if col.startswith("GB_"):
        print(col)

GB_temperature
GB_radiation_direct_horizontal
GB_radiation_diffuse_horizontal


### 5. Save only the columns I need

The two full files are too big for GitHub (over 100 MB each), so I save the three original columns this project uses in a small file, `gb_columns_raw.csv`.
Notebook 02 starts from this file. The values are not changed.

In [8]:
# read again only the 3 columns I need, as text, so the values stay exactly as in the original files
gb_load = pd.read_csv(load_file, usecols=["utc_timestamp", "GB_GBN_load_actual_entsoe_transparency"], dtype=str)
gb_weather = pd.read_csv(weather_file, usecols=["utc_timestamp", "GB_temperature"], dtype=str)

gb_columns = pd.merge(gb_load, gb_weather, on="utc_timestamp", how="left")
gb_columns.to_csv(raw_folder + "/gb_columns_raw.csv", index=False)

print("Saved gb_columns_raw.csv:", len(gb_columns), "rows")
gb_columns.head()

Saved gb_columns_raw.csv: 50401 rows


,utc_timestamp,GB_GBN_load_actual_entsoe_transparency,GB_temperature
0,2014-12-31T23:00:00Z,NaN,6.4250
1,2015-01-01T00:00:00Z,26758.0000,6.5650
2,2015-01-01T01:00:00Z,27166.0000,6.7150
3,2015-01-01T02:00:00Z,24472.0000,6.8930
4,2015-01-01T03:00:00Z,23003.0000,7.0930


### 6. Save the official forecast

The load file also has the official day-ahead forecast for Great Britain (`GB_GBN_load_forecast_entsoe_transparency`).
I do not use it as a feature. I only use it in notebook 10, to compare my models with the official forecast.

In [9]:
official = pd.read_csv(load_file, usecols=["utc_timestamp", "GB_GBN_load_forecast_entsoe_transparency"], dtype=str)
official.to_csv(raw_folder + "/gb_official_forecast_raw.csv", index=False)

print("Saved gb_official_forecast_raw.csv:", len(official), "rows")
official.head()

Saved gb_official_forecast_raw.csv: 50401 rows


,utc_timestamp,GB_GBN_load_forecast_entsoe_transparency
0,2014-12-31T23:00:00Z,NaN
1,2015-01-01T00:00:00Z,32057.0000
2,2015-01-01T01:00:00Z,31738.0000
3,2015-01-01T02:00:00Z,30690.0000
4,2015-01-01T03:00:00Z,28442.0000


### Summary

- Load file: 300 columns for 32 countries. For this project I need `GB_GBN_load_actual_entsoe_transparency`.
- Weather file: 85 columns for 28 countries. For this project I need `GB_temperature`.
- These two columns and the time column are saved in `datasets/raw/gb_columns_raw.csv`.
- The official forecast is saved in `datasets/raw/gb_official_forecast_raw.csv`. It is only used in notebook 10.

**Next:** notebook 02 - clean the data.